# PEMSD8 reproduction and seed check

**Goal.** Finish the PEMSD8 Quantile trio and test whether the physics loss's gain is bigger than run-to-run noise.

| Run label | Model | Seed | Why |
|---|---|---|---|
| `pemsd8_quantile_seed0` | Quantile (plain) | 0 | completes the trio for the Table 1 comparison |
| `pemsd8_gpde_quantile_seed0/1/2` | GPDE+Quantile | 0, 1, 2 | paired with the three GPDE+Quantile+Phy seeds already trained on the implementation workstream (`p2/results/horizon_metrics.csv`, λ = 0.002, same seeds, same initialisation) |

Each run is 50 epochs, about 1 hour on a T4. **We split the four runs over two Colab accounts** (see `MY_RUNS` below), and on each account run them one after the other. Two runs on one GPU are slower than sequential.

We ran sections 0–3, then section 4 trains and evaluates `MY_RUNS`.
When all four runs show `DONE`, run sections 0, 3 and 5 on either account.

## 0. Drive, settings, and which runs this account does

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, re, shutil, subprocess, threading, time
BASE  = '/content/drive/MyDrive/CS637'
AUTH  = f'{BASE}/uq4phy_traffic_network'
P1    = f'{BASE}/p1'
RUNDR = f'{P1}/pemsd8_runs'              # checkpoints, logs and DONE markers on Drive
OUT8  = f'{BASE}/outputs/pemsd8'         # prediction files (.npz)
RES   = f'{P1}/results'
DATA8 = '/content/data/PEMSD8'           # local disk, downloaded every session
SCRIPTS = '/content/p1_scripts'         # helper scripts run from local disk (Drive can show new files late)
for d in [RUNDR, OUT8, RES, SCRIPTS, f'{RES}/pemsd8_runs']:
    os.makedirs(d, exist_ok=True)

RUNS = {  # label -> (authors' folder, seed)
    'pemsd8_gpde_quantile_seed0': ('gpde_quantile_model_PEMSD8', 0),
    'pemsd8_quantile_seed0':      ('quantile_model_PEMSD8', 0),
    'pemsd8_gpde_quantile_seed1': ('gpde_quantile_model_PEMSD8', 1),
    'pemsd8_gpde_quantile_seed2': ('gpde_quantile_model_PEMSD8', 2),
}
#the same notebook was run on two differnet accounts for two different runs
# ---- choose per account ----
# account A: MY_RUNS = ['pemsd8_gpde_quantile_seed0', 'pemsd8_quantile_seed0']
# account B: MY_RUNS = ['pemsd8_gpde_quantile_seed1', 'pemsd8_gpde_quantile_seed2']
MY_RUNS = ['pemsd8_gpde_quantile_seed0', 'pemsd8_quantile_seed0']
SAVE_PREDICTIONS = True   # about 150 MB per run in outputs/pemsd8/; set False if Drive space is tight

# FRESH_START = True deletes THIS account's runs (MY_RUNS) from Drive and retrains them from scratch.
# Use it for the first Run all. If the session dies and you run again, set it to False first,
# or finished runs get deleted and retrained.
FRESH_START = True
CLEARED = set()
if FRESH_START:
    for r in MY_RUNS:
        for path in [f'{RUNDR}/{r}', f'{RES}/pemsd8_runs/{r}.csv', f'{OUT8}/{r}.npz']:
            try:
                shutil.rmtree(path) if os.path.isdir(path) else (os.remove(path) if os.path.exists(path) else None)
            except OSError as e:
                print('note: could not delete', path, e)
        CLEARED.add(r)
    print('FRESH_START: cleared', sorted(CLEARED), '- these will be retrained from scratch')
for r in RUNS:
    print(f'{r:30s}', 'DONE' if os.path.exists(f'{RUNDR}/{r}/DONE') else '-', '<- this account' if r in MY_RUNS else '')

Mounted at /content/drive
FRESH_START: cleared ['pemsd8_gpde_quantile_seed0', 'pemsd8_quantile_seed0'] - these will be retrained from scratch
pemsd8_gpde_quantile_seed0     - <- this account
pemsd8_quantile_seed0          - <- this account
pemsd8_gpde_quantile_seed1     - 
pemsd8_gpde_quantile_seed2     - 


In [2]:
import torch, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

torch 2.11.0+cu130 | cuda True
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 3 MiB, 15360 MiB


## 1. Authors' PEMSD8 folders

Notebook 1 already cloned these three folders. This cell only checks they're there.

In [3]:
for f in sorted({v[0] for v in RUNS.values()}):
    ok = os.path.exists(f'{AUTH}/{f}/dcrnn_train_pytorch.py')
    print(f'{f:32s}', 'ok' if ok else 'MISSING')
    if not ok:
        !git -C "{AUTH}" sparse-checkout add /{f}/

gpde_quantile_model_PEMSD8       ok
quantile_model_PEMSD8            ok


## 2. PEMSD8 data

These are the authors' processed files (train/val/test.npz) from figshare article 28347662, saved to Colab's local disk.

In [4]:
import requests, zipfile
def get_figshare(article_id, dst):
    os.makedirs(dst, exist_ok=True)
    files = requests.get(f'https://api.figshare.com/v2/articles/{article_id}', timeout=60).json()['files']
    for f in files:
        path = f'{dst}/{f["name"]}'
        if not (os.path.exists(path) and os.path.getsize(path) == f['size']):
            print('downloading', f['name'], f'{f["size"]/1e6:.0f} MB')
            with requests.get(f['download_url'], stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(path, 'wb') as out:
                    for chunk in r.iter_content(1 << 22): out.write(chunk)
        if path.endswith('.zip'):
            zipfile.ZipFile(path).extractall(dst)
    return dst

get_figshare(28347662, DATA8)
found = {s: sorted(glob.glob(f'{DATA8}/**/{s}.npz', recursive=True)) for s in ['train', 'val', 'test']}
DATA8 = os.path.dirname(found['test'][0])            # in case the files sit in a subfolder
shapes = {s: np.load(f'{DATA8}/{s}.npz')['x'].shape for s in ['train', 'val', 'test']}
print('DATA8 =', DATA8, shapes)
assert all(v[2:] == (170, 3) for v in shapes.values()), 'expected (N, 12, 170, 3)'

downloading train.npz 611 MB
downloading PEMSD8 data processing.py 0 MB
downloading pems8ndy.npy 73 MB
downloading val.npz 86 MB
downloading test.npz 174 MB
DATA8 = /content/data/PEMSD8 {'train': (12476, 12, 170, 3), 'val': (1762, 12, 170, 3), 'test': (3549, 12, 170, 3)}


## 3. Helper scripts (updated versions of the notebook 1 scripts)

Written to Colab's local disk every session (`/content/p1_scripts`) and copied to `p1/scripts/` on Drive (our project drive other than git) as a record.

Changes from notebook 1:
- `eval_checkpoint.py` takes `--config`, and re-enables the prediction output that the PEMSD8 variants comment out in `evaluate()`.
- Both scripts treat |truth| < 1e-3 as missing (finding 29).
- `horizon_metrics.py` can add the authors' batch padding (`pad_to_batch`) to match their reported numbers exactly.

Numbers from notebook 1 don't change: PEMSD4 has no missing readings.

In [5]:
%%writefile {SCRIPTS}/eval_checkpoint.py
"""Evaluate one of the authors' shipped checkpoints and save val + test predictions.

Run from INSIDE a variant folder of the authors' repo, e.g.
    cd .../uq4phy_traffic_network/gpde_quantile_phy_model_PEMSD4
    python /path/eval_checkpoint.py --data_dir /content/pemsd4 --out /path/model.npz
The authors' code is not modified: we load their pretrained config with yaml.safe_load
(their run_demo uses yaml.load(f), which fails on PyYAML 6) and let torch.load read
their full checkpoint dict (PyTorch >= 2.6 defaults to weights_only=True).
"""
import argparse, glob, json, os, sys, time
import numpy as np, torch, yaml

_torch_load = torch.load
def _load(*a, **k):
    k.setdefault('weights_only', False)
    return _torch_load(*a, **k)
torch.load = _load

sys.path.insert(0, os.getcwd())
from lib.utils import load_graph_data                      # noqa: E402  (authors' code)
from model.pytorch.dcrnn_supervisor import DCRNNSupervisor  # noqa: E402
import inspect, textwrap                                    # noqa: E402
import model.pytorch.dcrnn_supervisor as _sup               # noqa: E402

# In the PEMSD8 variants evaluate() builds the predictions but its return line ends in
# "loss_mae#,{'prediction': ...}" (commented out). Re-enable it here, without editing their file.
_src = inspect.getsource(DCRNNSupervisor.evaluate)
if "#,{'prediction'" in _src:
    _ns = {}
    exec(textwrap.dedent(_src.replace("#,{'prediction'", ",{'prediction'")), _sup.__dict__, _ns)
    DCRNNSupervisor.evaluate = _ns['evaluate']
    print('NOTE: re-enabled the prediction output of evaluate() (commented out in this variant)')

p = argparse.ArgumentParser()
p.add_argument('--data_dir', required=True)
p.add_argument('--out', required=True)
p.add_argument('--log_dir', default='/content/p1_eval_logs')
p.add_argument('--epoch', type=int, default=None, help='default: epoch in the pretrained config')
p.add_argument('--config', default=None, help='default: data/model/pretrained/*/config.yaml (shipped checkpoints)')
a = p.parse_args()

cfg_path = a.config or sorted(glob.glob('data/model/pretrained/*/config.yaml'))[0]
cfg = yaml.safe_load(open(cfg_path))
saved = sorted(int(f[3:-4]) for f in os.listdir('models') if f.startswith('epo') and f.endswith('.tar'))
epoch = a.epoch if a.epoch is not None else cfg['train']['epoch']
assert epoch in saved, f'epo{epoch}.tar not in models/ (have {saved})'
if epoch != max(saved):
    print(f'NOTE: config epoch {epoch} is not the latest saved checkpoint ({max(saved)})')
cfg['data']['dataset_dir'] = a.data_dir
cfg['train']['epoch'] = epoch
cfg['train']['log_dir'] = os.path.join(a.log_dir, os.path.basename(os.getcwd()))

_, _, adj = load_graph_data(cfg['data']['graph_pkl_filename'])
t0 = time.time()
sup = DCRNNSupervisor(adj_mx=adj, **cfg)
result, meta = {}, {'variant': os.path.basename(os.getcwd()), 'epoch': epoch, 'config': cfg_path,
                    'reg_lambda': cfg['model'].get('reg'), 'seed': os.environ.get('SEED'),
                    'test_batch_size': cfg['data'].get('test_batch_size'),
                    'device': str(next(sup.dcrnn_model.parameters()).device)}
for split in ['val', 'test']:
    mis, width, mse, rmse, mae, out = sup.evaluate(split)
    n = len(np.load(os.path.join(a.data_dir, split + '.npz'))['y'])
    pred = np.stack(out['prediction']).astype(np.float32)   # (12, N_padded, nodes, 3) = lower, median, upper
    truth = np.stack(out['truth']).astype(np.float32)       # (12, N_padded, nodes)
    near0 = (truth != 0) & (np.abs(truth) < 1e-3)           # missing readings come back as float residues
    meta[f'{split}_missing_share'] = float((np.abs(truth) < 1e-3).mean())
    truth[near0] = 0.0
    meta[f'{split}_samples'] = n
    meta[f'{split}_padded_samples'] = int(pred.shape[1])
    meta[f'{split}_authors_metrics'] = dict(mae=float(mae), mis=float(mis), width=float(width), mse=float(mse), rmse=float(rmse))
    result[f'{split}_pred'] = pred[:, :n]        # drop the copies of the last sample used to pad the final batch
    result[f'{split}_truth'] = truth[:, :n]
    print(f'{split}: authors-style (all 12 horizons, incl. padding)  MAE {mae:.4f}  MIS {mis:.4f}  '
          f'width {width:.4f}  MSE {mse:.4f}  RMSE {rmse:.4f}   [{n} samples, padded to {pred.shape[1]}]')
meta['seconds'] = round(time.time() - t0, 1)
os.makedirs(os.path.dirname(a.out), exist_ok=True)
np.savez_compressed(a.out, meta=json.dumps(meta), **result)
print('saved', a.out, f'({os.path.getsize(a.out)/1e6:.0f} MB, {meta["seconds"]} s)')

Writing /content/p1_scripts/eval_checkpoint.py


In [6]:
%%writefile {SCRIPTS}/horizon_metrics.py
"""Per-horizon metrics from the prediction files written by eval_checkpoint.py.

Two readings of "15 / 30 / 60 min", because the paper does not say which it uses:
  step : the single forecast step at that lead time (step 3, 6, 12 of 12)
  avg  : the average over all steps up to that lead time (steps 1-3, 1-6, 1-12);
         avg at 60 min is the all-horizon average the authors' code prints
MIS is reported two ways:
  mis_authors : the code's mis_loss = (u - l) + |y - median| + 40*(y - u)+ + 40*(l - y)+
  mis_std     : the paper's Eq. 4 / standard interval score, without the |y - median| term
Readings where truth == 0 are excluded, as in the authors' masked losses; |truth| < 1e-3 counts
as 0 too, because missing readings come back from evaluate() as tiny float residues.
pad_to_batch: add copies of the last sample the way the authors' DataLoader pads the final
batch (their reported numbers include these copies); use it only to match their numbers.
"""
import numpy as np

LEADS = {15: 3, 30: 6, 60: 12}           # minutes -> number of 5-minute steps
RHO = 2.0 / 0.05                         # 2 / alpha for a 95% interval

def _metrics(pred, truth):
    m = np.abs(truth) >= 1e-3
    lo, mid, hi, y = pred[..., 0][m], pred[..., 1][m], pred[..., 2][m], truth[m]
    lo, mid, hi, y = (v.astype(np.float64) for v in (lo, mid, hi, y))
    err = mid - y
    pen = RHO * np.maximum(y - hi, 0) + RHO * np.maximum(lo - y, 0)
    mse = np.mean(err ** 2)
    return dict(mae=np.mean(np.abs(err)), mse=mse, rmse=np.sqrt(mse),
                mis_authors=np.mean(hi - lo + np.abs(err) + pen), mis_std=np.mean(hi - lo + pen),
                width=np.mean(hi - lo), coverage=np.mean((y >= lo) & (y <= hi)),
                crossed=np.mean((lo > mid) | (mid > hi)))

def pad_like_authors(pred, truth, batch_size):
    n = pred.shape[1]; extra = (-n) % batch_size
    if extra == 0:
        return pred, truth
    return (np.concatenate([pred, np.repeat(pred[:, -1:], extra, axis=1)], axis=1),
            np.concatenate([truth, np.repeat(truth[:, -1:], extra, axis=1)], axis=1))

def horizon_table(pred, truth, pad_to_batch=None):
    """pred (12, N, nodes, 3), truth (12, N, nodes) -> list of dict rows."""
    if pad_to_batch:
        pred, truth = pad_like_authors(pred, truth, pad_to_batch)
    rows = []
    for minutes, k in LEADS.items():
        rows.append(dict(view='step', minutes=minutes, **_metrics(pred[k - 1], truth[k - 1])))
        rows.append(dict(view='avg', minutes=minutes, **_metrics(pred[:k], truth[:k])))
    return rows

Writing /content/p1_scripts/horizon_metrics.py


In [7]:
# keep a copy on Drive for the repo (best effort; the notebook itself uses the local copies)
for f in ['eval_checkpoint.py', 'horizon_metrics.py']:
    try:
        os.makedirs(f'{P1}/scripts', exist_ok=True); shutil.copy2(f'{SCRIPTS}/{f}', f'{P1}/scripts/{f}')
    except OSError as e:
        print('note: could not copy', f, 'to Drive yet:', e)
print('scripts ready in', SCRIPTS, os.listdir(SCRIPTS))

scripts ready in /content/p1_scripts ['eval_checkpoint.py', 'horizon_metrics.py']


## 4. Train and evaluate this account's runs

For each run, this cell:
1. Copies the authors' folder to Colab's local disk, without their checkpoints and logs.
2. Changes one line in our copy, `random_seed = 0` → the run's seed, and points the config at the local data.
3. Trains 50 epochs and prints one line per epoch. Every 2 minutes it copies checkpoints and logs to `p1/pemsd8_runs/<label>/` on Drive.
4. Marks the run `DONE`, then evaluates its best checkpoint (the last one saved, since a checkpoint is saved only when validation MAE improves) and writes per-horizon metrics.


In [8]:
sys.path.insert(0, SCRIPTS)
import importlib, horizon_metrics; importlib.reload(horizon_metrics)
import pandas as pd

def _epochs(d):
    return sorted(int(re.search(r'epo(\d+)\.tar', f).group(1)) for f in glob.glob(f'{d}/epo*.tar'))

def _copy_authors(folder, dst):
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(f'{AUTH}/{folder}', dst, ignore=shutil.ignore_patterns('models', 'dcrnn_DR_*', '__pycache__', '*.npz'))

def _retry(fn, what, tries=12, wait=15):
    # Drive (especially a shared folder used through a shortcut) can take a minute or two
    # before a new folder accepts files, so file operations on Drive are retried.
    last = None
    for _ in range(tries):
        try:
            return fn()
        except OSError as e:
            last = e; time.sleep(wait)
    print(f'WARNING: {what} still failing after {tries} tries: {last}')

def train(label):
    folder, seed = RUNS[label]
    ddir = f'{RUNDR}/{label}'
    if label not in CLEARED and os.path.exists(f'{ddir}/DONE'):
        print(f'{label}: already DONE on Drive, not retraining'); return
    if os.path.exists(f'{ddir}/DONE'):                              # cleared, but Drive still shows it: remove again
        try: os.remove(f'{ddir}/DONE')
        except OSError: pass
    os.makedirs(ddir, exist_ok=True)
    _retry(lambda: open(f'{ddir}/.write_test', 'w').write('ok'), 'first write to the Drive run folder')
    for old in glob.glob(f'{ddir}/epo*.tar'): os.remove(old)       # leftovers of an unfinished run: start clean
    local = f'/content/p1_runs/{label}'
    _copy_authors(folder, local)
    sp = f'{local}/model/pytorch/dcrnn_supervisor.py'; s = open(sp).read()
    assert s.count('random_seed = 0') == 1; open(sp, 'w').write(s.replace('random_seed = 0', f'random_seed = {seed}'))
    cp = f'{local}/data/model/dcrnn_la.yaml'; c = open(cp).read()
    assert 'dataset_dir: data/PEMSD8' in c; open(cp, 'w').write(c.replace('dataset_dir: data/PEMSD8', f'dataset_dir: {DATA8}'))
    stdout_path = f'/content/p1_runs/{label}.stdout.log'
    print(f'=== training {label} ({folder}, seed {seed}); about 1 h')

    def sync():
        for f in glob.glob(f'{local}/models/epo*.tar'):
            dst = f'{ddir}/{os.path.basename(f)}'
            if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f): shutil.copy2(f, dst)
        for f in glob.glob(f'{local}/data/model/dcrnn_DR_*/info.log'): shutil.copy2(f, f'{ddir}/info.log')
        if os.path.exists(stdout_path): shutil.copy2(stdout_path, f'{ddir}/stdout.log')
    stop = threading.Event()
    def loop():
        fails = 0
        while not stop.wait(120):
            try:
                sync(); fails = 0
            except OSError as e:
                fails += 1
                if fails == 3: print(f'(note: copying to Drive has failed 3 times in a row; training continues, files are safe on local disk: {e})')
    threading.Thread(target=loop, daemon=True).start()

    t0 = time.time()
    with open(stdout_path, 'w') as log:
        p = subprocess.Popen([sys.executable, 'dcrnn_train_pytorch.py', '--config_filename=data/model/dcrnn_la.yaml'],
                             cwd=local, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            if 'val_mae' in line or 'Traceback' in line or 'Error' in line:
                print(f'[{(time.time()-t0)/60:5.1f} min] ' + line.split(' - ')[-1].strip()[:150])
        p.wait()
    stop.set()
    if p.returncode != 0:
        print(open(stdout_path).read()[-4000:]); raise RuntimeError(f'{label} failed; see the output above')
    _retry(sync, 'final copy of checkpoints and logs to Drive', tries=10, wait=30)
    best = max(_epochs(f'{local}/models'))
    info = dict(best_epoch=best, minutes=round((time.time()-t0)/60, 1), seed=seed, folder=folder)
    _retry(lambda: open(f'{ddir}/DONE', 'w').write(json.dumps(info)), 'writing the DONE marker')
    print(f'{label}: DONE, best epoch {best}, {info["minutes"]:.0f} min')

def evaluate_run(label):
    folder, seed = RUNS[label]; ddir = f'{RUNDR}/{label}'
    run_models = f'/content/p1_runs/{label}/models'            # present if this session trained the run
    try:
        best = json.load(open(f'{ddir}/DONE'))['best_epoch']
    except OSError:
        best = max(_epochs(run_models))
    src = f'{run_models}/epo{best}.tar'
    if not os.path.exists(src): src = f'{ddir}/epo{best}.tar'
    local = f'/content/p1_eval/{label}'
    _copy_authors(folder, local)
    # the PEMSD8 variants save to models/ but load from <folder>/models/ (hard-coded): put the checkpoint in both
    for mdir in [f'{local}/models', f'{local}/{folder}/models']:
        os.makedirs(mdir, exist_ok=True); shutil.copy2(src, f'{mdir}/epo{best}.tar')
    npz = f'{OUT8}/{label}.npz' if SAVE_PREDICTIONS else f'/content/{label}.npz'
    r = subprocess.run([sys.executable, f'{SCRIPTS}/eval_checkpoint.py', '--config', 'data/model/dcrnn_la.yaml',
                        '--epoch', str(best), '--data_dir', DATA8, '--out', npz, '--log_dir', '/content/p1_eval_logs'],
                       cwd=local, capture_output=True, text=True, env={**os.environ, 'SEED': str(seed)})
    print('\n'.join(l for l in r.stdout.splitlines() if 'authors-style' in l or l.startswith('saved')))
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-5000:]); raise RuntimeError(f'evaluation of {label} failed; see the output above')
    z = np.load(npz); rows = []
    for split in ['val', 'test']:
        for pad in [None, 128]:
            for row in horizon_metrics.horizon_table(z[f'{split}_pred'], z[f'{split}_truth'], pad_to_batch=pad):
                if pad: row['view'] += '_padded'
                rows.append(dict(label=label, split=split, epoch=best, seed=seed, law='none', reg_lambda=None, **row))
    df = pd.DataFrame(rows)
    _retry(lambda: df.to_csv(f'{RES}/pemsd8_runs/{label}.csv', index=False), 'writing the metrics CSV')
    print(f'{label}: evaluated at epoch {best} -> {RES}/pemsd8_runs/{label}.csv')
    return df

for label in MY_RUNS:
    train(label)
    evaluate_run(label)

=== training pemsd8_gpde_quantile_seed0 (gpde_quantile_model_PEMSD8, seed 0); about 1 h
[  1.1 min] Epoch [0/50] (98) val_mae:3.4418, val_mis: 80.2145, val_width: 5.2382, val_mse: 42.0808, val_rmse: 6.4870 ,
[  2.1 min] Epoch [1/50] (196) val_mae:3.1370, val_mis: 68.3728, val_width: 5.0157, val_mse: 31.5973, val_rmse: 5.6211 ,
[  3.1 min] Epoch [2/50] (294) val_mae:2.7579, val_mis: 59.8619, val_width: 4.5711, val_mse: 29.4006, val_rmse: 5.4222 ,
[  4.1 min] Epoch [3/50] (392) val_mae:2.5828, val_mis: 49.5735, val_width: 4.8457, val_mse: 25.0436, val_rmse: 5.0044 ,
[  5.2 min] Epoch [4/50] (490) val_mae:2.3050, val_mis: 54.0858, val_width: 3.2785, val_mse: 22.2664, val_rmse: 4.7187 ,
[  6.2 min] Epoch [5/50] (588) val_mae:2.3096, val_mis: 50.1065, val_width: 3.6958, val_mse: 20.3797, val_rmse: 4.5144 ,
[  7.2 min] Epoch [6/50] (686) val_mae:2.0446, val_mis: 41.7818, val_width: 3.9490, val_mse: 19.2978, val_rmse: 4.3929 ,
[  8.2 min] Epoch [7/50] (784) val_mae:2.2831, val_mis: 48.7115, v

## 5. Analysis (after all four runs are DONE)

**Tables:**
- **A.** Our PEMSD8 trio vs Table 1, in both readings of "15/30/60 min".
- **B. The seed check.** GPDE+Quantile (3 seeds, ours) vs GPDE+Quantile+Phy (3 seeds, implementation workstream), paired by seed.
- **C.** How much of the headline gain comes from GPDE and how much from the physics loss.
- **D.** PEMSD4 again, with the authors' batch padding added back, to see exactly which reading Table 1 used.

In [14]:
sys.path.insert(0, SCRIPTS)
import importlib, horizon_metrics; importlib.reload(horizon_metrics)
import pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

missing = [r for r in RUNS if not os.path.exists(f'{RES}/pemsd8_runs/{r}.csv')]
READY = not missing
if not READY:
    print('Waiting for runs that are not evaluated yet:', missing)
    print('Run section 0 and section 5 again when all four are DONE.')
else:
    ours = pd.concat([pd.read_csv(f'{RES}/pemsd8_runs/{r}.csv') for r in RUNS])
    ours['model'] = ours.label.map(lambda l: 'Quantile' if l.startswith('pemsd8_quantile') else 'GPDE+Quantile')

    p2 = pd.read_csv(f'{BASE}/p2/results/horizon_metrics.csv')
    phy_labels = {'pemsd8_underwood_ours': 0, 'pemsd8_underwood_ours_seed1': 1, 'pemsd8_underwood_ours_seed2': 2}
    phy = p2[p2.label.isin(phy_labels)].copy()
    phy['seed'] = phy.label.map(phy_labels); phy['model'] = 'GPDE+Quantile+Phy'
    assert phy.groupby('label').size().min() > 0 and set(phy.seed) == {0, 1, 2}, 'Phy runs not found in p2/results/horizon_metrics.csv'
    phy = phy.drop_duplicates(['label', 'split', 'view', 'minutes'], keep='last')
    allr = pd.concat([ours, phy], ignore_index=True)
    allr = allr[allr.split == 'test']
    print(allr.groupby(['model', 'view']).seed.nunique().unstack())
    allr.to_csv(f'{RES}/pemsd8_all_runs_test.csv', index=False)

view               avg  avg_padded  step  step_padded
model                                                
GPDE+Quantile      3.0         3.0   3.0          3.0
GPDE+Quantile+Phy  3.0         NaN   3.0          NaN
Quantile           1.0         1.0   1.0          1.0


In [15]:
if not READY:
    print('Skipped: waiting for all four runs (see the message above).')
else:
    # A. Trio vs Table 1 (PEMSD8 columns: MAE, MIS, MSE, RMSE)
    T1_8 = {
     'Quantile':          {15: (1.202, 13.738, 6.541, 2.557), 30: (1.386, 19.878, 9.453, 3.074), 60: (1.666, 30.333, 14.622, 3.823)},
     'GPDE+Quantile':     {15: (1.041, 12.633, 4.732, 2.175), 30: (1.251, 19.200, 7.730, 2.780), 60: (1.540, 29.760, 12.714, 3.565)},
     'GPDE+Quantile+Phy': {15: (1.024, 12.063, 4.676, 2.162), 30: (1.236, 18.497, 7.696, 2.774), 60: (1.521, 28.812, 12.524, 3.539)},
    }
    mets = ['mae', 'mis_authors', 'mse', 'rmse', 'mis_std', 'width', 'coverage']
    g = allr.groupby(['model', 'view', 'minutes'])[mets]
    A = g.mean().join(g.std().add_suffix('_std')).join(allr.groupby(['model', 'view', 'minutes']).seed.nunique().rename('n_seeds')).reset_index()
    for i, k in enumerate(['mae', 'mis_authors', 'mse', 'rmse']):
        A[f'T1_{k}'] = [T1_8[m][mi][i] for m, mi in zip(A.model, A.minutes)]
        A[f'{k}_diff_%'] = 100 * (A[k] - A[f'T1_{k}']) / A[f'T1_{k}']
    A.round(4).to_csv(f'{RES}/pemsd8_trio_vs_table1.csv', index=False)
    A['abs_diff'] = A[['mae_diff_%', 'mis_authors_diff_%', 'mse_diff_%']].abs().mean(axis=1)
    print('Mean |% diff| vs Table 1 (MAE, MIS, MSE) for each reading of 15/30/60 min:')
    display(A.pivot_table(index='model', columns=['view', 'minutes'], values='abs_diff').round(2))
    display(A[A.view.isin(['step', 'avg'])][['model', 'view', 'minutes', 'n_seeds', 'mae', 'mae_std', 'T1_mae', 'mae_diff_%',
            'mis_authors', 'T1_mis_authors', 'mis_authors_diff_%', 'rmse', 'T1_rmse', 'coverage']].round(3))

Mean |% diff| vs Table 1 (MAE, MIS, MSE) for each reading of 15/30/60 min:


view                 avg             avg_padded               step               step_padded              
minutes               15    30    60         15    30    60     15     30     60          15     30     60
model                                                                                                     
GPDE+Quantile       0.86  0.97  1.24       0.98  1.15  1.55  33.40  45.13  49.48       32.92  44.40  48.57
GPDE+Quantile+Phy   4.13  2.64  2.02        NaN   NaN   NaN  39.74  49.39  54.08         NaN    NaN    NaN
Quantile           11.16  8.98  6.60      11.18  8.78  6.22  38.52  49.80  56.91       38.37  49.21  56.00

,model,view,minutes,n_seeds,mae,mae_std,T1_mae,mae_diff_%,mis_authors,T1_mis_authors,mis_authors_diff_%,rmse,T1_rmse,coverage
0,GPDE+Quantile,avg,15,3,1.044,0.023,1.041,0.322,12.385,12.633,-1.965,2.172,2.175,0.873
1,GPDE+Quantile,avg,30,3,1.255,0.024,1.251,0.327,18.732,19.200,-2.439,2.778,2.780,0.804
2,GPDE+Quantile,avg,60,3,1.542,0.022,1.540,0.147,28.704,29.760,-3.549,3.566,3.565,0.724
6,GPDE+Quantile,step,15,3,1.234,0.025,1.041,18.547,17.162,12.633,35.849,2.626,2.175,0.806
7,GPDE+Quantile,step,30,3,1.564,0.025,1.251,24.989,28.614,19.200,49.029,3.532,2.780,0.709
8,GPDE+Quantile,step,60,3,2.001,0.023,1.540,29.913,45.231,29.760,51.987,4.601,3.565,0.607
12,GPDE+Quantile+Phy,avg,15,3,1.060,0.015,1.024,3.480,12.581,12.063,4.297,2.212,2.162,0.882
13,GPDE+Quantile+Phy,avg,30,3,1.268,0.016,1.236,2.621,19.060,18.497,3.045,2.805,2.774,0.807
14,GPDE+Quantile+Phy,avg,60,3,1.555,0.016,1.521,2.242,29.414,28.812,2.090,3.569,3.539,0.715
15,GPDE+Quantile+Phy,step,15,3,1.249,0.018,1.024,21.999,17.528,12.063,45.307,2.665,2.162,0.810


In [16]:
if not READY:
    print('Skipped: waiting for all four runs (see the message above).')
else:
    # B. Seed check: is the physics gain bigger than seed-to-seed noise?  (paired by seed)
    rows, prs = [], []
    for view in ['step', 'avg']:
        for mnt in [15, 30, 60]:
            for k in ['mae', 'rmse', 'mis_authors', 'mis_std', 'coverage']:
                a = allr[(allr.model == 'GPDE+Quantile') & (allr.view == view) & (allr.minutes == mnt)].set_index('seed')[k]
                b = allr[(allr.model == 'GPDE+Quantile+Phy') & (allr.view == view) & (allr.minutes == mnt)].set_index('seed')[k]
                d = (b - a).dropna()
                rows.append(dict(view=view, minutes=mnt, metric=k, gpde_mean=a.mean(), gpde_std=a.std(), phy_mean=b.mean(), phy_std=b.std(),
                                 diff_mean=d.mean(), diff_pct=100 * d.mean() / a.mean(), paired_diffs=' / '.join(f'{x:+.3f}' for x in d),
                                 same_sign_all_seeds=bool((d > 0).all() or (d < 0).all()),
                                 gap_over_noise=abs(d.mean()) / max(a.std(), b.std())))
    B = pd.DataFrame(rows).round(4)
    B.to_csv(f'{RES}/pemsd8_seed_check.csv', index=False)
    display(B[B.view == 'step'])
    print('\ngap_over_noise = |mean paired difference| / larger seed std. Below about 1, the physics effect is within run-to-run noise.')
    print('"Lower is better" for mae, rmse, mis; "higher is better" for coverage. diff = Phy minus GPDE, per seed 0 / 1 / 2.')

,view,minutes,metric,gpde_mean,gpde_std,phy_mean,phy_std,diff_mean,diff_pct,paired_diffs,same_sign_all_seeds,gap_over_noise
0,step,15,mae,1.2341,0.0251,1.2493,0.0175,0.0152,1.2313,+0.026 / +0.031 / -0.012,False,0.6045
1,step,15,rmse,2.6265,0.0339,2.6652,0.0235,0.0387,1.4728,+0.058 / +0.050 / +0.007,True,1.1412
2,step,15,mis_authors,17.1618,0.3132,17.5284,0.5912,0.3666,2.1362,+0.148 / +1.234 / -0.283,False,0.6201
3,step,15,mis_std,15.9277,0.2935,16.2791,0.5744,0.3514,2.2062,+0.122 / +1.203 / -0.271,False,0.6118
4,step,15,coverage,0.8057,0.0177,0.8097,0.0064,0.0039,0.4872,+0.020 / -0.022 / +0.014,False,0.2214
5,step,30,mae,1.5636,0.0253,1.5741,0.0163,0.0105,0.6727,+0.019 / +0.033 / -0.021,False,0.4151
6,step,30,rmse,3.5318,0.0173,3.5437,0.0125,0.0118,0.3350,+0.005 / +0.034 / -0.004,False,0.6857
7,step,30,mis_authors,28.6136,0.6203,29.1589,0.2188,0.5453,1.9058,+0.380 / +1.188 / +0.068,True,0.8791
8,step,30,mis_std,27.0500,0.6190,27.5848,0.2281,0.5348,1.9770,+0.361 / +1.154 / +0.089,True,0.8639
9,step,30,coverage,0.7087,0.0233,0.7014,0.0185,-0.0072,-1.0227,-0.004 / -0.015 / -0.002,True,0.3117



gap_over_noise = |mean paired difference| / larger seed std. Below about 1, the physics effect is within run-to-run noise.
"Lower is better" for mae, rmse, mis; "higher is better" for coverage. diff = Phy minus GPDE, per seed 0 / 1 / 2.


In [17]:
if not READY:
    print('Skipped: waiting for all four runs (see the message above).')
else:
    # C. Where does the headline gain come from?  (paper: 14.8% at 15 min = 13.4 points GPDE + 1.4 points physics)
    m = allr.groupby(['model', 'view', 'minutes']).mae.mean()
    for view in ['avg', 'step']:
        for mnt in [15, 30, 60]:
            q, gq, gp = m[('Quantile', view, mnt)], m[('GPDE+Quantile', view, mnt)], m[('GPDE+Quantile+Phy', view, mnt)]
            print(f'{view:4s} {mnt:2d} min: MAE {q:.3f} -> {gq:.3f} -> {gp:.3f}   total {100*(q-gp)/q:5.1f}%  '
                  f'= GPDE {100*(q-gq)/q:5.1f} + physics {100*(gq-gp)/q:5.1f} points')
    t = T1_8; print(f'paper 15 min: {t["Quantile"][15][0]} -> {t["GPDE+Quantile"][15][0]} -> {t["GPDE+Quantile+Phy"][15][0]}')
    print('note: plain Quantile is one seed; the other two are 3-seed means')

avg  15 min: MAE 1.276 -> 1.044 -> 1.060   total  16.9%  = GPDE  18.1 + physics  -1.2 points
avg  30 min: MAE 1.448 -> 1.255 -> 1.268   total  12.4%  = GPDE  13.3 + physics  -0.9 points
avg  60 min: MAE 1.717 -> 1.542 -> 1.555   total   9.4%  = GPDE  10.2 + physics  -0.7 points
step 15 min: MAE 1.420 -> 1.234 -> 1.249   total  12.0%  = GPDE  13.1 + physics  -1.1 points
step 30 min: MAE 1.712 -> 1.564 -> 1.574   total   8.1%  = GPDE   8.7 + physics  -0.6 points
step 60 min: MAE 2.173 -> 2.001 -> 2.017   total   7.2%  = GPDE   7.9 + physics  -0.8 points
paper 15 min: 1.202 -> 1.041 -> 1.024
note: plain Quantile is one seed; the other two are 3-seed means


In [13]:
# D. PEMSD4 with the authors' batch padding (test batch 256) added back: which reading did Table 1 use?
T1_4 = {
 'Quantile':          {15: (2.373, 42.771, 24.356), 30: (2.419, 45.320, 25.316), 60: (2.670, 56.975, 30.868)},
 'GPDE+Quantile':     {15: (1.641, 25.043, 10.929), 30: (1.722, 27.897, 12.321), 60: (2.133, 42.622, 20.418)},
 'GPDE+Quantile+Phy': {15: (1.539, 22.179, 10.231), 30: (1.621, 24.744, 11.706), 60: (2.018, 37.807, 19.834)},
 'MAEMIS':            {15: (2.326, 39.423, 16.708), 30: (2.382, 42.429, 18.002), 60: (3.167, 56.240, 25.347)},
 'GPDE+MAEMIS':       {15: (1.633, 24.601, 10.971), 30: (1.708, 27.488, 12.392), 60: (2.098, 43.089, 20.731)},
 'GPDE+MAEMIS+Phy':   {15: (1.359, 16.962, 7.422),  30: (1.623, 24.807, 11.652), 60: (2.044, 38.501, 20.882)},
}
P4 = {'Quantile': 'quantile_model_PEMSD4', 'GPDE+Quantile': 'gpde_quantile_model_PEMSD4', 'GPDE+Quantile+Phy': 'gpde_quantile_phy_model_PEMSD4',
      'MAEMIS': 'maemis_model_PEMSD4', 'GPDE+MAEMIS': 'gpde_maemis_model_PEMSD4', 'GPDE+MAEMIS+Phy': 'gpde_maemis_phy_model_PEMSD4_0.005'}
rows = []
for name, folder in P4.items():
    z = np.load(f'{BASE}/outputs/pemsd4/{folder}.npz')
    for pad in [None, 256]:
        for r in horizon_metrics.horizon_table(z['test_pred'], z['test_truth'], pad_to_batch=pad):
            t = T1_4[name][r['minutes']]
            rows.append(dict(model=name, view=r['view'] + ('_padded' if pad else ''), minutes=r['minutes'],
                             mae=r['mae'], T1_mae=t[0], mis=r['mis_authors'], T1_mis=t[1], mse=r['mse'], T1_mse=t[2]))
D = pd.DataFrame(rows)
for k in ['mae', 'mis', 'mse']: D[f'{k}_diff_%'] = 100 * (D[k] - D[f'T1_{k}']) / D[f'T1_{k}']
D.round(4).to_csv(f'{RES}/pemsd4_padded_vs_table1.csv', index=False)
D['abs_diff'] = D[['mae_diff_%', 'mis_diff_%', 'mse_diff_%']].abs().mean(axis=1)
print('PEMSD4: mean |% diff| vs Table 1. Near 0 = the reading Table 1 used.')
display(D.pivot_table(index='model', columns=['view', 'minutes'], values='abs_diff').round(2))

PEMSD4: mean |% diff| vs Table 1. Near 0 = the reading Table 1 used.


view                 avg             avg_padded               step               step_padded              
minutes               15    30    60         15    30    60     15     30     60          15     30     60
model                                                                                                     
GPDE+MAEMIS        18.35  2.85  3.39      20.08  0.20  0.15   3.56  48.85  61.23        1.07  44.22  55.26
GPDE+MAEMIS+Phy     2.16  2.96  3.64       0.25  0.05  0.09  32.71  51.26  66.95       28.97  46.05  60.08
GPDE+Quantile      18.57  3.07  3.69      20.51  0.11  0.17   4.35  49.11  60.51        1.45  44.11  54.11
GPDE+Quantile+Phy  21.18  2.31  2.93      22.45  0.08  0.04   3.64  51.12  59.26        1.55  46.82  53.83
MAEMIS             10.73  3.37  7.13      13.44  0.05  5.13   6.63  32.57  34.84        3.60  28.07  29.62
Quantile            6.67  2.93  3.09       9.18  0.04  0.03   6.43  22.49  31.70        3.43  18.72  27.39